In [3]:
import tkinter as tk
from tkinter import ttk
import random

# =========================
# IMPORT ALGORITHMS
# =========================

from Algorithm.HutBui_BFS_1 import BFS, get_path
from Algorithm.HutBui_BFS_2 import BFS as BFS_2, get_path as get_path_2
from Algorithm.HutBui_DFS_1 import DFS, get_path as get_path_dfs
from Algorithm.HutBui_DFS_2 import DFS as DFS_2, get_path as get_path_dfs_2
from Algorithm.HutBui_IDS_1 import IDS, get_path as get_path_ids
from Algorithm.HutBui_IDS_2 import IDS as IDS_2, get_path as get_path_ids_2
from Algorithm.HutBui_UCS import UCS, get_path as get_path_ucs
from Algorithm.HutBui_Greedy import Greedy, get_path as get_path_greedy
from Algorithm.HutBui_AStar import AStar, get_path as get_path_astar
from Algorithm.HutBui_IDAStar import IDAStar, get_path as get_path_idastar
from Algorithm.HutBui_SimpleHill import SimpleHill, get_path as get_path_simplehill
from Algorithm.HutBui_SteppestHill import SteppestHill, get_path as get_path_steepesthill
from Algorithm.HutBui_StochasticHill import StochasticHill, get_path as get_path_stochastichill
from Algorithm.HutBui_RandomRestartHill import RandomRestartHill, get_path as get_path_randomrestarthill
from Algorithm.HutBui_Beam import Beam, get_path as get_path_beam
from Algorithm.HutBui_SimulatedAnnealing import SimulatedAnnealing, get_path as get_path_simulatedannealing
from Algorithm.HutBui_UnobservableSearch import UnobservableSearch, build_unobservable_belief, get_path as get_path_unobservable
from Algorithm.HutBui_PartialSearch import PartialSearch, build_partial_belief, get_path as get_path_partial
from Algorithm.HutBui_AndOrSearch import AND_OR_Search, get_path as get_path_andor

# =========================
# CONSTANT
# =========================

CELL_SIZE = 60
DELAY = 500

path = []
current_step = 0
after_id = None
rows = 0
cols = 0
original_matrix = []
visited_cells = set()

belief_canvases = []
last_belief_states = []
belief_visited_cells = []
belief_finished = []
belief_count = 0

# =========================
# WINDOW
# =========================

root = tk.Tk()

root.title("Vacuum Cleaner AI Visualizer")

root.geometry("1400x900")

root.configure(bg="#f5f5f5")

# =========================
# STYLE
# =========================

style = ttk.Style()

style.theme_use("clam")

style.configure(
    "TCombobox",
    fieldbackground="white",
    background="white",
    foreground="#333333"
)

# =========================
# TITLE
# =========================

title = tk.Label(
    root,
    text="Vacuum Cleaner AI Visualizer",
    bg="#f5f5f5",
    fg="#222222",
    font=("Segoe UI", 26, "bold")
)

title.pack(pady=15)

# =========================
# TOP FRAME
# =========================

top_frame = tk.Frame(
    root,
    bg="white",
    bd=1,
    relief="solid"
)

top_frame.pack(
    fill="x",
    padx=20,
    pady=10
)

# =========================
# ROW INPUT
# =========================

tk.Label(
    top_frame,
    text="Rows",
    bg="white",
    fg="#333333",
    font=("Segoe UI", 11)
).pack(side="left", padx=10, pady=10)

rows_entry = tk.Entry(
    top_frame,
    width=5,
    font=("Segoe UI", 11),
    bg="#fafafa",
    fg="#333333",
    relief="solid",
    bd=1
)

rows_entry.pack(side="left")

rows_entry.insert(0, "5")

# =========================
# COL INPUT
# =========================

tk.Label(
    top_frame,
    text="Cols",
    bg="white",
    fg="#333333",
    font=("Segoe UI", 11)
).pack(side="left", padx=10)

cols_entry = tk.Entry(
    top_frame,
    width=5,
    font=("Segoe UI", 11),
    bg="#fafafa",
    fg="#333333",
    relief="solid",
    bd=1
)

cols_entry.pack(side="left")

cols_entry.insert(0, "5")

# =========================
# ALGORITHM
# =========================

algorithm_var = tk.StringVar()

algorithm_menu = ttk.Combobox(
    top_frame,
    textvariable=algorithm_var,
    values=[
        "BFS 1",
        "BFS 2",
        "DFS 1",
        "DFS 2",
        "IDS 1",
        "IDS 2",
        "UCS",
        "Greedy",
        "A*",
        "IDA*",
        "Simple Hill",
        "Steppest Hill",
        "Stochastic Hill",
        "Random Restart Hill",
        "Beam",
        "Simulated Annealing",
        "Unobservable Search",
        "Partial Search",
        "AND-OR Search"
    ],
    width=15,
    state="readonly",
    font=("Segoe UI", 11)
)

algorithm_menu.current(0)

algorithm_menu.pack(
    side="left",
    padx=20
)

# =========================
# BEAM WIDTH INPUT
# =========================

beam_width_label = tk.Label(
    top_frame,
    text="Beam Width",
    bg="white",
    fg="#333333",
    font=("Segoe UI", 11)
)

beam_width_label.pack(side="left", padx=10)

beam_width_entry = tk.Entry(
    top_frame,
    width=5,
    font=("Segoe UI", 11),
    bg="#fafafa",
    fg="#333333",
    relief="solid",
    bd=1
)

beam_width_entry.pack(side="left")

beam_width_entry.insert(0, "2")

# =========================
# ESTIMATE MATRICES INPUT
# =========================

estimate_count_label = tk.Label(
    top_frame,
    text="Estimate Matrices",
    bg="white",
    fg="#333333",
    font=("Segoe UI", 11)
)

estimate_count_label.pack(side="left", padx=10)

estimate_count_entry = tk.Entry(
    top_frame,
    width=5,
    font=("Segoe UI", 11),
    bg="#fafafa",
    fg="#333333",
    relief="solid",
    bd=1
)

estimate_count_entry.pack(side="left")

estimate_count_entry.insert(0, "2")


# =========================
# START BUTTON
# =========================

start_btn = tk.Button(
    top_frame,
    text="START",
    bg="#4CAF50",
    fg="white",
    activebackground="#43a047",
    activeforeground="white",
    font=("Segoe UI", 11, "bold"),
    relief="flat",
    cursor="hand2",
    padx=15,
    pady=5
)

start_btn.pack(
    side="left",
    padx=20
)

# =========================
# CONTENT FRAME
# =========================

content_frame = tk.Frame(
    root,
    bg="#f5f5f5"
)

content_frame.pack(
    fill="both",
    expand=True,
    padx=20,
    pady=10
)

# =========================
# TOP CONTENT
# =========================

top_content = tk.Frame(
    content_frame,
    bg="#f5f5f5"
)

top_content.pack()

# =========================
# MATRIX PANEL
# =========================

matrix_frame = tk.Frame(
    top_content,
    bg="white",
    bd=1,
    relief="solid"
)

matrix_frame.pack(
    side="left",
    padx=10
)

matrix_title = tk.Label(
    matrix_frame,
    text="Input Matrix",
    bg="white",
    fg="#222222",
    font=("Segoe UI Semibold", 14)
)

matrix_title.pack(pady=10)

# =========================
# MATRIX INPUT
# =========================

matrix_input = tk.Text(
    matrix_frame,
    width=20,
    height=8,
    font=("Consolas", 12),
    bg="#fafafa",
    fg="#333333",
    relief="solid",
    bd=1
)

matrix_input.pack(
    padx=20,
    pady=20
)

matrix_input.insert(
    tk.END,
    "1 1 1 1 1\n1 0 1 0 1\n1 1 1 1 1\n0 1 0 1 0\n1 1 1 1 1"
)

# =========================
# MAP PANEL
# =========================

map_frame = tk.Frame(
    top_content,
    bg="white",
    bd=1,
    relief="solid"
)

map_frame.pack(
    side="right",
    padx=10
)

canvas = tk.Canvas(
    map_frame,
    width=600,
    height=600,
    bg="white",
    highlightthickness=0
)

canvas.pack(
    padx=20,
    pady=20
)

belief_canvas_frame = tk.Frame(
    map_frame,
    bg="white"
)

# =========================
# LOG PANEL
# =========================

log_frame = tk.Frame(
    content_frame,
    bg="white",
    bd=1,
    relief="solid"
)

log_frame.pack(
    fill="both",
    expand=True,
    pady=10,
    padx=10
)

log_title = tk.Label(
    log_frame,
    text="Log",
    bg="white",
    fg="#222222",
    font=("Segoe UI Semibold", 14)
)

log_title.pack(pady=10)

# =========================
# LOG CONTAINER
# =========================

log_container = tk.Frame(
    log_frame,
    bg="white"
)

log_container.pack(
    fill="both",
    expand=True,
    padx=20,
    pady=10
)

# =========================
# SCROLLBAR
# =========================

scrollbar = tk.Scrollbar(
    log_container
)

scrollbar.pack(
    side="right",
    fill="y"
)

# =========================
# LOG BOX
# =========================

log_box = tk.Text(
    log_container,
    height=18,
    bg="white",
    fg="#333333",
    font=("Consolas", 11),
    relief="flat",
    yscrollcommand=scrollbar.set
)

log_box.pack(
    side="left",
    fill="both",
    expand=True
)

scrollbar.config(
    command=log_box.yview
)

# =========================
# DRAW MAP
# =========================

def draw_map_on_canvas(target_canvas, state, robot_x=None, robot_y=None, visited=None, cell_size=CELL_SIZE):

    target_canvas.delete("all")

    for r in range(rows):
        for c in range(cols):

            x1 = c * cell_size
            y1 = r * cell_size

            x2 = x1 + cell_size
            y2 = y1 + cell_size

            if visited is not None and (r, c) in visited:
                color = "#ffeaa7"

            elif state[r][c] == 1:
                color = "#ff6b6b"

            else:
                color = "#b7efc5"

            target_canvas.create_rectangle(
                x1,
                y1,
                x2,
                y2,
                fill=color,
                outline="#dddddd",
                width=2
            )

    if robot_x is not None and robot_y is not None:

        padding = max(4, cell_size // 5)

        x1 = robot_y * cell_size + padding
        y1 = robot_x * cell_size + padding

        x2 = x1 + cell_size - 2 * padding
        y2 = y1 + cell_size - 2 * padding

        target_canvas.create_oval(
            x1,
            y1,
            x2,
            y2,
            fill="#4A90E2",
            outline="white",
            width=2
        )


def draw_map(state):
    draw_map_on_canvas(
        canvas,
        state,
        visited=visited_cells,
        cell_size=CELL_SIZE
    )

def clear_belief_canvases():

    for widget in belief_canvas_frame.winfo_children():
        widget.destroy()

    belief_canvases.clear()
    belief_visited_cells.clear()
    belief_finished.clear()
    last_belief_states.clear()


def setup_belief_canvases(count):

    canvas.pack_forget()

    clear_belief_canvases()

    last_belief_states.clear()

    belief_canvas_frame.pack(
        padx=10,
        pady=10
    )

    small_cell_size = 35

    for i in range(count):

        case_frame = tk.Frame(
            belief_canvas_frame,
            bg="white",
            bd=1,
            relief="solid"
        )

        case_frame.grid(
            row=i // 3,
            column=i % 3,
            padx=8,
            pady=8
        )

        title = tk.Label(
            case_frame,
            text=f"Khả năng {i + 1}",
            bg="white",
            fg="#222222",
            font=("Segoe UI", 10, "bold")
        )

        title.pack(pady=3)

        small_canvas = tk.Canvas(
            case_frame,
            width=cols * small_cell_size,
            height=rows * small_cell_size,
            bg="white",
            highlightthickness=0
        )

        small_canvas.pack(
            padx=5,
            pady=5
        )

        belief_canvases.append(small_canvas)
        belief_visited_cells.append(set())
        last_belief_states.append(None)
        belief_finished.append(False)


def show_normal_canvas():

    belief_canvas_frame.pack_forget()

    clear_belief_canvases()

    canvas.pack(
        padx=20,
        pady=20
    )

# =========================
# AUTO RUN
# =========================

def auto_run():

    global current_step
    global after_id

    if current_step >= len(path):

        log_box.insert(
            tk.END,
            "\nFinished!\n"
        )

        log_box.see(tk.END)

        after_id = None

        return

    data = path[current_step]

    if "belief_state" in data:

        belief_state = data["belief_state"]

        if len(belief_canvases) == 0:
            setup_belief_canvases(belief_count)

        action = data["action"]

        if action is None:
            action = "START"

        log_box.insert(
            tk.END,
            f"Step {current_step}: {action}\n"
        )

        for index in range(belief_count):

            if belief_finished[index]:
                if last_belief_states[index] is None:
                    continue

                room, x, y = last_belief_states[index]

            else:
                if index < len(belief_state):
                    room, x, y = belief_state[index]

                    if sum(row.count(1) for row in room) == 0:
                        belief_finished[index] = True

                    last_belief_states[index] = (room, x, y)
                    belief_visited_cells[index].add((x, y))

                else:
                    if last_belief_states[index] is None:
                        continue

                    room, x, y = last_belief_states[index]

            draw_map_on_canvas(
                belief_canvases[index],
                [list(row) for row in room],
                x,
                y,
                belief_visited_cells[index],
                cell_size=35
            )

            status = "DONE" if belief_finished[index] else "RUN"

            log_box.insert(
                tk.END,
                f"  Khả năng {index + 1}: robot=({x},{y}) - {status}\n"
            )

        log_box.insert(tk.END, "\n")
        log_box.see(tk.END)

        current_step += 1

        after_id = root.after(DELAY, auto_run)

        return

    x = data["x"]

    y = data["y"]

    state = data["state"]

    visited_cells.add((x, y))

    draw_map(state)

    # =========================
    # ROBOT
    # =========================

    x1 = y * CELL_SIZE + 12

    y1 = x * CELL_SIZE + 12

    x2 = x1 + 36

    y2 = y1 + 36

    canvas.create_oval(
        x1,
        y1,
        x2,
        y2,
        fill="#4A90E2",
        outline="white",
        width=2
    )

    action = data["action"]

    if action is None:
        action = "START"

    if "step" in data:

        info = f"Step {data['step']}"

    elif "depth" in data:

        info = f"Depth {data['depth']}"

    elif "cost" in data:

        info = f"Cost {data['cost']}"

    else:

        info = ""

    if algorithm_var.get() == "AND-OR Search":

        predicted_action = data.get("action")
        real_action = data.get("real_action")

        if predicted_action is None:
            predicted_action = "START"

        if real_action is None:
            real_action = "START"

        log_box.insert(
            tk.END,
            f"Step {data.get('step', current_step)}\n"
        )

        log_box.insert(
            tk.END,
            f"  Hành động dự đoán: {predicted_action}\n"
        )

        log_box.insert(
            tk.END,
            f"  Hành động thực hiện: {real_action}\n"
        )

        log_box.insert(
            tk.END,
            f"  Vị trí máy hút bụi: ({x},{y})\n"
        )

        log_box.insert(
            tk.END,
            f"  Số ô dơ còn lại: {data.get('cost', '')}\n\n"
        )

    else:

        log_box.insert(
            tk.END,
            f"{info}: {action} -> ({x},{y})\n"
        )

    log_box.see(tk.END)

    current_step += 1

    after_id = root.after(DELAY, auto_run)

def belief_path_to_gui_path(belief_path):
    gui_path = []

    for data in belief_path:
        belief_state = data["state"]

        room, x, y = belief_state[0]

        gui_path.append({
            "action": data["action"],
            "state": [list(row) for row in room],
            "x": x,
            "y": y,
            "cost": data["cost"],
            "g": data["g"],
            "belief_state": belief_state
        })

    return gui_path

# =========================
# START ALGORITHM
# =========================

def start_algorithm():

    global rows
    global cols
    global original_matrix
    global path
    global current_step
    global visited_cells
    global after_id
    global belief_count

    if after_id is not None:
        root.after_cancel(after_id)
        after_id = None

    visited_cells.clear()

    current_step = 0

    log_box.delete(
        1.0,
        tk.END
    )

    rows = int(rows_entry.get())

    cols = int(cols_entry.get())

    selected_algorithm = algorithm_var.get()

    if selected_algorithm in ["Unobservable Search", "Partial Search"]:
        try:
            belief_count = int(estimate_count_entry.get())
        except:
            log_box.insert(tk.END, "Estimate Matrices phải là số nguyên.\n")
            return

        if belief_count < 1:
            belief_count = 1
    else:
        belief_count = 0


    if selected_algorithm in ["Unobservable Search", "Partial Search"]:

        original_matrix = [[0] * cols for _ in range(rows)]

    else:

        matrix_text = matrix_input.get(
            1.0,
            tk.END
        ).strip()

        matrix_lines = matrix_text.split("\n")

        matrix = []

        for line in matrix_lines:

            matrix.append(
                list(map(int, line.split()))
            )

        original_matrix = [row[:] for row in matrix]


    if selected_algorithm in ["Unobservable Search", "Partial Search"]:

        clear_belief_canvases()
        canvas.pack_forget()

    else:

        show_normal_canvas()

        canvas.config(
            width=cols * CELL_SIZE,
            height=rows * CELL_SIZE
        )

        draw_map(original_matrix)

    goal_state = [[0] * cols for _ in range(rows)]

    robot_x = random.randint(0, rows - 1)

    robot_y = random.randint(0, cols - 1)

    result = None

    # BFS 1

    if selected_algorithm == "BFS 1":

        result = BFS(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path(result) if result else []

    # BFS 2

    elif selected_algorithm == "BFS 2":

        result = BFS_2(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_2(result) if result else []

    # DFS 1

    elif selected_algorithm == "DFS 1":

        result = DFS(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_dfs(result) if result else []

    # DFS 2

    elif selected_algorithm == "DFS 2":

        result = DFS_2(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_dfs_2(result) if result else []

    # IDS 1

    elif selected_algorithm == "IDS 1":

        depth_limit = 0

        while result is None:

            result = IDS(
                [row[:] for row in original_matrix],
                goal_state,
                robot_x,
                robot_y,
                depth_limit
            )

            depth_limit += 1

        path = get_path_ids(result)

    # IDS 2

    elif selected_algorithm == "IDS 2":

        depth_limit = 0

        while result is None:

            result = IDS_2(
                [row[:] for row in original_matrix],
                goal_state,
                robot_x,
                robot_y,
                depth_limit
            )

            depth_limit += 1

        path = get_path_ids_2(result)

    # UCS

    elif selected_algorithm == "UCS":

        result = UCS(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_ucs(result) if result else []

    # GREEDY

    elif selected_algorithm == "Greedy":

        result = Greedy(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_greedy(result) if result else []

    # A*

    elif selected_algorithm == "A*":

        result = AStar(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_astar(result) if result else []
    
    # IDA*

    elif selected_algorithm == "IDA*":

        result = IDAStar(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_idastar(result) if result else []

    # SIMPLE HILL

    elif selected_algorithm == "Simple Hill":

        result = SimpleHill(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_simplehill(result) if result else []


    # STEPPEST HILL

    elif selected_algorithm == "Steppest Hill":

        result = SteppestHill(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_steepesthill(result) if result else []

    # STOCHASTIC HILL

    elif selected_algorithm == "Stochastic Hill":

        result = StochasticHill(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_stochastichill(result) if result else []
    
    # RANDOM RESTART HILL

    elif selected_algorithm == "Random Restart Hill":
        
        result = RandomRestartHill(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        path = get_path_randomrestarthill(result) if result else []

    # BEAM
    elif selected_algorithm == "Beam":

        beam_width = int(beam_width_entry.get())

        result = Beam(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y,
            beam_width
        )

        if result:
            path = get_path_beam(result)
        else:
            path = []
    
    # SIMULATED ANNEALING
    elif selected_algorithm == "Simulated Annealing":

        result = SimulatedAnnealing(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y
        )

        if result:
            path = get_path_simulatedannealing(result)
        else:
            path = []

    # UNOBSERVABLE SEARCH
    elif selected_algorithm == "Unobservable Search":

        estimate_count = belief_count

        base_matrix = [
            [random.randint(0, 1) for _ in range(cols)]
            for _ in range(rows)
        ]

        initial_belief = build_unobservable_belief(
            base_matrix,
            robot_x,
            robot_y,
            estimate_count
        )

        setup_belief_canvases(belief_count)

        log_box.insert(tk.END, f"Running {selected_algorithm}\n")
        log_box.insert(tk.END, f"Robot Start Position: ({robot_x},{robot_y})\n")
        log_box.insert(tk.END, "Đang tạo các ma trận dự đoán...\n\n")
        log_box.see(tk.END)

        for index in range(min(belief_count, len(initial_belief))):
            room, x, y = initial_belief[index]

            last_belief_states[index] = (room, x, y)
            belief_visited_cells[index].add((x, y))

            draw_map_on_canvas(
                belief_canvases[index],
                [list(row) for row in room],
                x,
                y,
                belief_visited_cells[index],
                cell_size=35
            )

        root.update()

        result = UnobservableSearch(
            initial_belief,
            goal_state,
            rows,
            cols
        )

        if result:
            belief_path = get_path_unobservable(result)
            path = belief_path_to_gui_path(belief_path)
        else:
            path = []


    # PARTIAL SEARCH
    elif selected_algorithm == "Partial Search":

        estimate_count = belief_count

        initial_belief = build_partial_belief(
            rows,
            cols,
            robot_x,
            robot_y,
            estimate_count
        )

        result = PartialSearch(
            initial_belief,
            goal_state,
            rows,
            cols
        )

        if result:
            belief_path = get_path_partial(result)
            path = belief_path_to_gui_path(belief_path)
        else:
            path = []

        # AND-OR SEARCH
    elif selected_algorithm == "AND-OR Search":

        result, logs = AND_OR_Search(
            [row[:] for row in original_matrix],
            goal_state,
            robot_x,
            robot_y,
            max_depth=rows * cols * 5
        )

        if result:
            path = get_path_andor(result)
        else:
            path = []

    # LOG

    if selected_algorithm not in ["Unobservable Search", "Partial Search"]:
        log_box.insert(
            tk.END,
            f"Running {selected_algorithm}\n"
        )

        log_box.insert(
            tk.END,
            f"Robot Start Position: ({robot_x},{robot_y})\n\n"
        )

    if not path:
        log_box.insert(
            tk.END,
            "No Solution Found.\n"
        )

        return

    after_id = root.after(DELAY, auto_run)

# =========================
# BUTTON COMMAND
# =========================

start_btn.config(
    command=start_algorithm
)

# =========================
# MAIN LOOP
# =========================

root.mainloop()